# XTTS-v2: Voice Cloning from a 6-Second Clip

## What is XTTS-v2?
XTTS-v2 (by Coqui AI) can **clone any voice** from a short audio reference (6-30 seconds).  
Give it a recording of someone speaking → it generates new speech in that voice.

### Architecture
```
Reference Audio ──▶ [Speaker Encoder] ──▶ Speaker Embedding
     (6-30s)          (extracts voice                │
                       characteristics)              │
                                                     ▼
Text to speak ──▶ [GPT-2 Decoder] ──▶ [HiFi-GAN Vocoder] ──▶ 🔊 Audio
                  (autoregressive)    (tokens → waveform)
```

**Key insight**: The speaker encoder creates a fixed-size vector that captures:
- Pitch range, speaking rate, accent
- Voice timbre, breathiness, resonance

This embedding is then used to condition the GPT-2 decoder.

**17 languages** supported | **Voice cloning** | **Best overall quality**

---
**Kaggle Setup**: GPU T4 x2 | Attach dataset for reference audio

## Step 1: Install & Load

In [ ]:
# coqui-tts is the community fork that supports Python 3.12+
# (original TTS==0.22.0 only works on Python <3.12)
# pypinyin needed for Chinese, cutlet for Japanese
!pip install -q coqui-tts pypinyin cutlet
!pip install -q --force-reinstall 'protobuf>=5.26.1'

In [ ]:
import torch
import time
import os
from pathlib import Path
from IPython.display import Audio, display

# Auto-accept Coqui TOS for non-commercial use (required for batch mode)
os.environ["COQUI_TOS_AGREED"] = "1"

from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load XTTS-v2 (downloads ~1.8GB on first run)
print("Loading XTTS-v2...")
t0 = time.time()
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print(f"Loaded in {time.time()-t0:.1f}s")

## Step 2: Generate with Built-in Speakers

In [ ]:
# XTTS-v2 always requires a reference speaker — either speaker_wav or speaker_id
# Let's check what's available
print("Available speakers:", getattr(tts, 'speakers', None))
print("Available languages:", getattr(tts, 'languages', None))

In [ ]:
# XTTS-v2 needs a reference audio clip to clone the voice.
# Let's create one using Google TTS (gTTS) as a quick bootstrap.
# In real use, you'd record your own voice or use an existing clip.

!pip install -q gtts
from gtts import gTTS

ref_text = (
    "This is a sample reference recording. I am speaking naturally "
    "so the model can learn my voice characteristics and clone them."
)
gtts_obj = gTTS(text=ref_text, lang='en', slow=False)
gtts_obj.save("reference_voice_raw.mp3")

# Convert to WAV (XTTS needs WAV format)
import subprocess
subprocess.run(["ffmpeg", "-y", "-i", "reference_voice_raw.mp3", "-ar", "22050", "-ac", "1", "reference_voice.wav"],
               capture_output=True)

print("Reference audio created (via Google TTS):")
display(Audio("reference_voice.wav"))

# Now generate with XTTS using this reference
text = "Hello! I am XTTS version 2. I can clone any voice from just a short audio clip."

t0 = time.time()
tts.tts_to_file(text=text, speaker_wav="reference_voice.wav", language="en", file_path="xtts_default.wav")
print(f"\nXTTS output (cloned from reference) — Generated in {time.time()-t0:.1f}s:")
display(Audio("xtts_default.wav"))

## Step 3: Voice Cloning

This is the killer feature. You provide a reference WAV and XTTS speaks in that voice.

### Tips for good reference audio:
- **Duration**: 6-30 seconds (10-15s is ideal)
- **Quality**: Clean recording, no background noise
- **Content**: Normal conversational speech
- **Format**: WAV, 22050 Hz or higher

In [ ]:
import numpy as np

# Voice cloning: use the reference we created above
# In practice, upload YOUR voice recording here instead!
print("Using reference_voice.wav for cloning...")
print("(Replace with your own recording for real voice cloning)")
display(Audio("reference_voice.wav"))

In [ ]:
# Now clone that voice for new text!
clone_texts = [
    "This is voice cloning in action. I should sound like the reference.",
    "The weather today is perfect for a walk in the park.",
    "Machine learning makes this kind of technology possible.",
]

for i, text in enumerate(clone_texts):
    out_path = f"cloned_{i}.wav"
    t0 = time.time()
    tts.tts_to_file(
        text=text,
        speaker_wav="reference_voice.wav",  # <-- the magic!
        language="en",
        file_path=out_path,
    )
    print(f"\n[{time.time()-t0:.1f}s] {text}")
    display(Audio(out_path))

## Step 4: Cross-lingual Voice Cloning

The most impressive feature: clone a voice in **a different language** than the reference!

Example: English reference → speak Vietnamese, Chinese, French...

In [ ]:
# Cross-lingual: English voice → other languages
# Supported: en, es, fr, de, it, pt, pl, tr, ru, nl, cs, ar, zh-cn, hu, ko, ja, hi
# Note: Vietnamese (vi) is NOT supported by XTTS-v2

cross_lingual = [
    ("Chinese",    "zh-cn", "你好！这是从英语克隆到中文的声音。非常神奇。"),
    ("French",     "fr", "Bonjour! Ceci est un clonage vocal de l'anglais vers le français."),
    ("Japanese",   "ja", "こんにちは！これは英語から日本語への音声クローンです。"),
    ("Spanish",    "es", "¡Hola! Esta es una clonación de voz del inglés al español."),
    ("Korean",     "ko", "안녕하세요! 이것은 영어에서 한국어로의 음성 복제입니다."),
]

print("Cross-lingual voice cloning (English reference → other languages):")
print("="*60)

for lang_name, lang_code, text in cross_lingual:
    out_path = f"cross_lingual_{lang_code}.wav"
    t0 = time.time()
    tts.tts_to_file(
        text=text,
        speaker_wav="reference_voice.wav",
        language=lang_code,
        file_path=out_path,
    )
    print(f"\n🌍 {lang_name} [{time.time()-t0:.1f}s]")
    print(f"   {text}")
    display(Audio(out_path))

## Step 5: Quality Settings

In [ ]:
# XTTS-v2 has some generation parameters you can tune
# Access the underlying model for more control

from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Show current config
print("Generation config:")
print(f"  Temperature: {getattr(tts.synthesizer.tts_model, 'temperature', 'default')}")
print(f"  Length penalty: {getattr(tts.synthesizer.tts_model, 'length_penalty', 'default')}")
print(f"  Repetition penalty: {getattr(tts.synthesizer.tts_model, 'repetition_penalty', 'default')}")

# You can also use tts_with_vc for voice conversion (different approach)
print("\nSupported languages:")
if hasattr(tts, 'languages'):
    print(tts.languages)

## Step 6: Batch Processing

In [ ]:
import re

def clone_voice_long(tts, text, speaker_wav, language="en", pause_sec=0.4):
    """Generate long speech with cloned voice."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    
    chunks = []
    sr = tts.synthesizer.output_sample_rate
    silence = np.zeros(int(sr * pause_sec))
    
    for i, sent in enumerate(sentences):
        print(f"  [{i+1}/{len(sentences)}] {sent[:60]}")
        wav = tts.tts(
            text=sent,
            speaker_wav=speaker_wav,
            language=language,
        )
        chunks.append(np.array(wav))
        chunks.append(silence)
    
    full = np.concatenate(chunks)
    print(f"\nTotal: {len(full)/sr:.1f}s")
    display(Audio(full, rate=sr))
    return full

story = (
    "Once upon a time, in a world powered by artificial intelligence, "
    "a small model learned to speak like any human. "
    "It could whisper in French, shout in Spanish, and sing in Japanese. "
    "The world was never the same again."
)

print("Generating story with cloned voice...")
_ = clone_voice_long(tts, story, "reference_voice.wav", "en")

## Key Takeaways

**Pros:**
- Best voice cloning quality from minimal data (6s!)
- 17 languages, cross-lingual cloning
- Can fine-tune for even better results (see `04_xtts_finetune/`)

**Cons:**
- Slowest of the 3 models tested
- Requires reference audio
- Coqui TTS library maintenance uncertain (company shut down)

**When to use**: Voice cloning, audiobook production, multilingual dubbing

### Model Comparison Summary

| Feature | Parler-TTS | Bark | XTTS-v2 |
|---------|-----------|------|----------|
| Voice control | Text description | Presets | Reference audio |
| Languages | English | 13+ | 17 |
| Voice cloning | No | No | **Yes** |
| Expressiveness | Medium | **High** | Medium |
| Speed | **Fast** | Slow | Slow |
| Fine-tuning | Possible | No | **Easy** |
| Quality | Good | Good | **Best** |